# 03 — Official business activity (test)

The Ministry of Commerce publishes the **registered activities of every commercial registration**
("ارتباط السجلات التجارية القائمة مع التصنيف الوطني للأنشطة الاقتصادية 2025", open.data.gov.sa).
Each row is one activity of one registration: national ID, section (level 1), ISIC class (level 2, 4 digits)
and SSIC activity (level 3, 6 digits). A business usually has several activities.

**Question of this notebook:** can these official codes replace (or check) the SSIC codes that
`01_preprocessing` guesses from the business name?

1. **Coverage:** how many of our businesses are in the file?
2. **Leakage check:** if the file lists only businesses still active in 2025, businesses that closed are
   missing from it, so "found in the file" would mean "survived". Coverage must be about the same for
   survived and disappeared businesses.
3. **Main activity:** the file has no "main activity" column, so two rules are compared: the section most of
   the activities belong to (`main_*`) and the first activity listed (`first_*`).
4. **Accuracy of the name classifier**, measured against the official codes.

If it works, the cells move into `01_preprocessing` Part 5.

**Needs:** `ssic_by_business.csv` from `01_preprocessing` Cell 19 (run 01 first on the **same Colab server**,
or upload the file to `DATA_DIR`), and the activity file in the Drive folder Darb DataSets.
**Rules:** raw files never changed; results saved in the data folder, never in Git; clear outputs before committing.

### Cell 1 — Setup

In [ ]:
import os
import gc
import re
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    %pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
else:
    DATA_DIR = Path(os.environ.get('DARB_DATA', str(Path.home() / 'darb-data')))

print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR)

### Cell 2 — The files
- **Activity file:** `CR_ISIC_2025.csv` (4.2 GB). A zipped copy (`CR_ISIC_2025.zip`) works too and uploads
  ~10× faster: change `ISIC_FILE` to the zip name. **Fill in `ISIC_DRIVE_ID`** with its Google Drive ID
  (the long code in its share link).
- **SSIC file:** the official list, to check the codes.
- **`ssic_by_business.csv`:** from `01_preprocessing` (one row per business: cohort, label, name-based SSIC code).

In [ ]:
ISIC_FILE = 'CR_ISIC_2025.csv'      # or 'CR_ISIC_2025.zip'
ISIC_DRIVE_ID = ''                  # <- fill in: Drive ID of the file in Darb DataSets
ISIC_PATH = DATA_DIR / 'MC_ISIC' / ISIC_FILE

SSIC_FILE = 'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx'
SSIC_PATH = DATA_DIR / 'Gastat' / SSIC_FILE
LABELS_PATH = DATA_DIR / 'ssic_by_business.csv'

if IN_COLAB:
    import gdown
    for path, drive_id in [(ISIC_PATH, ISIC_DRIVE_ID), (SSIC_PATH, '1wlS-84Ojsiv0UP8sjmZmNcENU_M8J0wZ')]:
        if not path.exists() and drive_id:
            path.parent.mkdir(parents=True, exist_ok=True)
            print('downloading', path.name, '(the activity file takes several minutes)')
            gdown.download(id=drive_id, output=str(path), quiet=True)

for path in [ISIC_PATH, SSIC_PATH, LABELS_PATH]:
    print(f'{path.name:<75}', f'found ({path.stat().st_size / 1e6:,.0f} MB)' if path.exists() else 'MISSING')

if not ISIC_PATH.exists():
    raise FileNotFoundError('Activity file missing: upload it to Darb DataSets and fill in ISIC_DRIVE_ID.')
if not LABELS_PATH.exists():
    raise FileNotFoundError('ssic_by_business.csv missing: run 01_preprocessing (Cells 1 to 19) on the same '
                            'Colab server first, or upload the file to ' + str(DATA_DIR))

### Cell 3 — Read the activity file (in pieces, only the codes)
The file is too big to read at once, so it is read 2 million rows at a time, keeping only the ID and the three
codes as numbers (small in memory). Each row keeps its position (`order`), so "the first activity listed" is known.
Rows with a missing or non-numeric code are counted, not guessed.

In [ ]:
NAMES = ['id', 'cr_number', 'section_code', 'section_desc', 'class_code', 'class_desc', 'activity_code', 'activity_desc']
CODES = ['id', 'section_code', 'class_code', 'activity_code']

parts, section_descs, rows, bad = [], set(), 0, 0
reader = pd.read_csv(ISIC_PATH, names=NAMES, header=0, usecols=CODES + ['section_desc'],
                     dtype=str, encoding='utf-8-sig', chunksize=2_000_000)
for chunk in reader:
    section_descs |= set(chunk[['section_code', 'section_desc']].drop_duplicates().itertuples(index=False, name=None))
    part = chunk[CODES].apply(lambda column: pd.to_numeric(column.str.strip(), errors='coerce'))
    ok = part.notna().all(axis=1)
    bad += int((~ok).sum())
    part = part[ok].astype({'id': 'int64', 'section_code': 'int8', 'class_code': 'int16', 'activity_code': 'int32'})
    part['order'] = part.index.to_numpy(dtype='int64')
    parts.append(part)
    rows += len(chunk)
    print(f'  read {rows:>12,} rows')

act = pd.concat(parts, ignore_index=True)
del parts, reader
gc.collect()
act['division'] = (act['class_code'] // 100).astype('int8')     # 4-digit class -> 2-digit division (0111 -> 01)

print()
print(f'Activity rows: {rows:,} | rows with a missing or wrong code (left out): {bad:,}')
print(f'Businesses (IDs): {act["id"].nunique():,} | memory: {act.memory_usage(deep=True).sum() / 1e9:.2f} GB')

### Cell 4 — Do the codes match the official SSIC?
- **Sections:** the file numbers them 1–21. SSIC uses Arabic letters, which `01_preprocessing` maps to A–U in
  their order. The table shows each number with the file's own name and the SSIC name of the letter: they must
  describe the same section.
- **Divisions:** every (section, division) pair must exist in SSIC.

In [ ]:
LETTERS = np.array(list('ABCDEFGHIJKLMNOPQRSTU'))

# The official SSIC list (same reading as 01_preprocessing Cell 16)
raw = pd.read_excel(SSIC_PATH, engine='calamine', header=None, dtype=str)
header_row = raw.index[raw[0].fillna('').str.strip() == 'رمز الباب'][0]
ssic = raw.iloc[header_row + 1:, :4].copy()
ssic.columns = ['section_code', 'section_name', 'division_code', 'division_name']
ssic = ssic.dropna(subset=['division_code']).apply(lambda column: column.str.strip())
arabic_sections = list(dict.fromkeys(ssic['section_code']))
ssic['letter'] = ssic['section_code'].map(dict(zip(arabic_sections, LETTERS)))
section_name = ssic.drop_duplicates('letter').set_index('letter')['section_name']
official_pairs = set(zip(ssic['letter'], ssic['division_code'].astype(int)))
del raw

def squash(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

codes = pd.DataFrame(sorted(section_descs, key=lambda p: int(p[0])), columns=['code', 'name in the file'])
codes['letter'] = LETTERS[codes['code'].astype(int) - 1]
codes['SSIC name of the letter'] = codes['letter'].map(section_name)
codes['same name'] = [squash(a) == squash(b) for a, b in zip(codes['name in the file'], codes['SSIC name of the letter'])]
with pd.option_context('display.width', 250, 'display.max_colwidth', 70):
    print(codes.to_string(index=False))

# (section, division) as one number: section number x 100 + division, e.g. C14 -> 314
official_keys = [(int(np.where(LETTERS == letter)[0][0]) + 1) * 100 + division for letter, division in official_pairs]
pair_ok = (act['section_code'].astype('int32') * 100 + act['division']).isin(official_keys).to_numpy()
print()
print(f'Activity rows whose (section, division) is in SSIC: {pair_ok.mean():.2%}')
if not pair_ok.all():
    wrong = act.loc[~pair_ok, ['section_code', 'division']].value_counts().head(10)
    print('Most common pairs not in SSIC (section number, division):')
    print(wrong.to_string())
del pair_ok

### Cell 5 — One row per business
For each ID:
- `n_activities`, `n_sections`, `sections_all` (e.g. `C|G`)
- **`main_section`**: the section most of its activities belong to (a tie goes to the one listed first);
  `main_division`: the most common division inside that section
- **`first_section`, `first_division`, `first_activity`**: the first activity listed in the file

In [ ]:
act = act.sort_values('order', kind='stable').reset_index(drop=True)
by_id = act.groupby('id', sort=False)
business = pd.DataFrame({'n_activities': by_id.size()})

first = act.drop_duplicates('id').set_index('id')
business['first_section'] = LETTERS[first['section_code'].reindex(business.index).to_numpy() - 1]
business['first_division'] = first['division'].reindex(business.index).astype(int).astype(str).str.zfill(2)
business['first_activity'] = first['activity_code'].reindex(business.index).astype(int).astype(str).str.zfill(6)

# Main section: most activities, tie -> listed first
per_section = (act.groupby(['id', 'section_code'], sort=False)
               .agg(n=('order', 'size'), first=('order', 'min')).reset_index())
business['n_sections'] = per_section.groupby('id').size().reindex(business.index)
bits = per_section.assign(bit=np.left_shift(1, per_section['section_code'].astype('int64') - 1)).groupby('id')['bit'].sum()
mask_text = {m: '|'.join(LETTERS[[i for i in range(21) if m >> i & 1]]) for m in bits.unique()}
business['sections_all'] = bits.reindex(business.index).map(mask_text)
main = (per_section.sort_values(['id', 'n', 'first'], ascending=[True, False, True])
        .drop_duplicates('id').set_index('id')['section_code'])
business['main_section'] = LETTERS[main.reindex(business.index).to_numpy() - 1]

# Main division: most common division inside the main section, tie -> listed first
per_division = (act.groupby(['id', 'section_code', 'division'], sort=False)
                .agg(n=('order', 'size'), first=('order', 'min')).reset_index())
per_division = per_division[per_division['section_code'].to_numpy() == main.reindex(per_division['id']).to_numpy()]
main_division = (per_division.sort_values(['id', 'n', 'first'], ascending=[True, False, True])
                 .drop_duplicates('id').set_index('id')['division'])
business['main_division'] = main_division.reindex(business.index).astype(int).astype(str).str.zfill(2)
del by_id, first, per_section, per_division, bits, main, main_division
gc.collect()

print(f'Businesses: {len(business):,}')
print()
print('Activities per business:')
print(business['n_activities'].describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())
print()
print(f'Businesses with activities in more than one section: {(business["n_sections"] > 1).mean():.1%}')
print(f'Main section = first section: {(business["main_section"] == business["first_section"]).mean():.1%}')
print()
print('Main section of the businesses:')
print(business['main_section'].map(lambda l: f'{l} {section_name.get(l, "")}').value_counts().to_string())

### Cell 6 — Coverage, and the leakage check
Joins the businesses of `01_preprocessing` (`entity_id`) to the file's IDs.

**Leakage check:** in the labelled cohorts, the share found in the file should be about the same for
**survived** and **disappeared** businesses. If disappeared businesses are found much less often, the file lists
only businesses still active at its date: then "found in the file" leaks the label, and the official code can
only be used where it does not depend on survival (for example as an answer key to improve the name
classifier), not directly as a model feature.

The coverage by start quarter also hints at the file's date: businesses that started after it are not in it.

In [ ]:
labels = pd.read_csv(LABELS_PATH, dtype=str, encoding='utf-8-sig', keep_default_na=False)
labels['id_number'] = pd.to_numeric(labels['entity_id'], errors='coerce').fillna(-1).astype('int64')
labels['found'] = labels['id_number'].isin(business.index)
labelled = labels[labels['cohort_quality'] != 'cannot_judge']
OUTCOME = {'1': 'survived', '0': 'disappeared', '': 'undecided'}

print(f'All businesses: {labels["found"].mean():.1%} found in the activity file ({labels["found"].sum():,} of {len(labels):,})')
print(f'Labelled cohorts: {labelled["found"].mean():.1%} found ({labelled["found"].sum():,} of {len(labelled):,})')
print()
print('Coverage per labelled cohort:')
print(labelled.groupby(['cohort', 'cohort_quality'])['found'].agg(businesses='size', found='mean')
      .assign(found=lambda t: t['found'].map('{:.1%}'.format)).to_string())
print()
by_outcome = labelled.groupby(labelled['survived_1y'].map(OUTCOME))['found'].agg(businesses='size', found='mean')
print('Coverage by outcome (the leakage check):')
print(by_outcome.assign(found=lambda t: t['found'].map('{:.1%}'.format)).to_string())
gap = by_outcome.loc['survived', 'found'] - by_outcome.loc['disappeared', 'found']
print()
print(f'Gap survived - disappeared: {gap:+.1%}')
print('⚠ Large gap: the file likely lists only businesses active at its date -> "found" leaks the label.'
      if gap > 0.10 else '✓ Small gap: being in the file does not depend much on survival.')
print('(rule of thumb: a gap above 10 points is a problem)')
print()
recent = labels[(labels['cohort'] >= '2023Q1') & (labels['cohort'] <= '2026Q4')]
print('Coverage by start quarter (hints at the date of the file):')
print(recent.groupby('cohort')['found'].agg(businesses='size', found='mean')
      .assign(found=lambda t: t['found'].map('{:.1%}'.format)).to_string())

### Cell 7 — How accurate is the name classifier?
For labelled businesses found in the file, the name-based code (`ssic_section`, `ssic_division` from
`01_preprocessing`) is compared with the official one, per `ssic_flag`:
- **= main / = first:** same section as the main / first official activity
- **in any:** the name's section is one of the business's official sections
- **division (ok only):** same division as the main official division

It also shows how many businesses the name classifier could not classify get an official code.

In [ ]:
both = labelled[labelled['found']].join(business, on='id_number')
has_section = both[both['ssic_section'] != '']
in_any = [s in a.split('|') for s, a in zip(has_section['ssic_section'], has_section['sections_all'])]
section_check = (has_section.assign(**{
                     '= main': has_section['ssic_section'] == has_section['main_section'],
                     '= first': has_section['ssic_section'] == has_section['first_section'],
                     'in any': in_any})
                 .groupby('ssic_flag')[['= main', '= first', 'in any']].mean())
section_check.insert(0, 'businesses', has_section['ssic_flag'].value_counts())
print('Section agreement, name classifier vs official (labelled businesses found in the file):')
print(section_check.to_string(formatters={c: '{:.1%}'.format for c in ['= main', '= first', 'in any']}))

ok_div = both[(both['ssic_flag'] == 'ok') & (both['ssic_division'] != '')]
print()
print(f'Division agreement (flag ok, {len(ok_div):,} businesses): '
      f'= main {(ok_div["ssic_division"].str.zfill(2) == ok_div["main_division"]).mean():.1%} | '
      f'= first {(ok_div["ssic_division"].str.zfill(2) == ok_div["first_division"]).mean():.1%}')
print()
no_code = labelled[labelled['ssic_flag'].isin(['unclassified', 'ambiguous', 'names_disagree'])]
print(f'Labelled businesses without a sure name-based section: {len(no_code):,} | '
      f'found in the activity file: {no_code["found"].mean():.1%}')

### Cell 8 — Save
Saves `official_activity_by_business.csv` (one row per ID in the activity file) in the data folder, never in Git.

In [ ]:
OUT = DATA_DIR / 'official_activity_by_business.csv'
business.to_csv(OUT, index_label='id', encoding='utf-8-sig')
print(f'Saved {len(business):,} businesses to {OUT}')

### Notes — results and decision (fill in after running)
- File date (from Cell 6):
- Coverage of labelled businesses:
- Leakage gap (survived − disappeared):
- Main-activity rule chosen (`main_*` or `first_*`), and why:
- Name classifier accuracy against the official codes:
- Decision: use the official codes in `01_preprocessing` Part 5? As a feature, or only as an answer key?